# graduate_agent_data 基线评估

使用**原模型**（无微调）在 graduate_agent_data 测试集上推理，评估：
- **意图识别准确率**：模型是否选择了正确的工具名
- **参数解析准确率**：模型生成的 arguments 是否与标注一致

## 1. 环境与依赖

In [ ]:
# 魔搭 Notebook 通常已预装，若缺失可取消注释安装
# !pip install ms-swift modelscope -q

## 2. 加载测试集

In [ ]:
from modelscope.msdatasets import MsDataset
import json

ds = MsDataset.load('yejintao/graduate_agent_data', split='test', cache_dir='./download_cache')
test_data = list(ds)
print(f'测试集样本数: {len(test_data)}')
print('首条样本 keys:', test_data[0].keys())
print('首条 user 内容:', test_data[0]['messages'][0]['content'][:80], '...')

## 3. 解析工具调用

从模型输出中解析 `<tool_call>...</tool_call>` 格式，提取 `name` 和 `arguments`。

In [ ]:
import re

def parse_tool_call_from_response(response: str):
    """
    从模型回复中解析第一个 tool_call。
    支持格式：<tool_call>\n{...}\n</tool_call>
    返回 (name, arguments) 或 (None, None)
    """
    if not response or not isinstance(response, str):
        return None, None
    match = re.search(r'<tool_call>\s*([\s\S]*?)\s*</tool_call>', response)
    if not match:
        return None, None
    try:
        content = match.group(1).strip()
        obj = json.loads(content)
        name = obj.get('name')
        args = obj.get('arguments', {})
        if isinstance(args, str):
            args = json.loads(args) if args.strip() else {}
        return name, args
    except Exception:
        return None, None


def get_ground_truth_tool_call(messages):
    """从 messages 中提取第一个 tool_call 作为标注。"""
    for m in messages:
        if m.get('role') == 'tool_call':
            content = m.get('content', '')
            try:
                obj = json.loads(content)
                name = obj.get('name')
                args = obj.get('arguments', {})
                if isinstance(args, str):
                    args = json.loads(args) if args.strip() else {}
                return name, args
            except Exception:
                return None, None
    return None, None


def normalize_args(args):
    """标准化参数便于比较：排序 key、统一类型。"""
    if not isinstance(args, dict):
        return {}
    out = {}
    for k, v in sorted(args.items()):
        if isinstance(v, str):
            out[k] = v.strip()
        elif isinstance(v, (int, float, bool)) or v is None:
            out[k] = v
        elif isinstance(v, list):
            out[k] = sorted(v) if all(isinstance(x, str) for x in v) else v
        else:
            out[k] = v
    return out


def _value_equal(pv, gv):
    """比较单个值是否等价。"""
    if isinstance(gv, str) and isinstance(pv, str):
        return pv.replace('T', ' ').strip() == gv.replace('T', ' ').strip()
    if isinstance(gv, (int, float)) and isinstance(pv, (int, float)):
        return abs(float(pv) - float(gv)) <= 1e-6
    return pv == gv


def args_match(pred_args, gt_args):
    """
    比较参数是否一致。
    允许 pred 省略可选参数（如 sample_size）或添加可选参数，只比较公共键的值。
    这样 GT={} 而 pred={include_target_stats:True, include_sample:True} 会判为正确。
    """
    p = normalize_args(pred_args)
    g = normalize_args(gt_args)
    common = set(p.keys()) & set(g.keys())
    for k in common:
        if not _value_equal(p.get(k), g.get(k)):
            return False
    return True

## 4. 初始化推理引擎

In [ ]:
# 模型选择：Qwen3-VL 需 transformers>=4.57，若报错可改用 Qwen2.5-7B
MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
# MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'  # 备选：纯文本模型

from swift.llm import PtEngine

# 原模型，不加载 adapters
engine = PtEngine(model_id_or_path=MODEL_ID)
print(f'引擎初始化完成: {MODEL_ID}')

## 5. 批量推理与评估

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 4  # 按需调整，避免 OOM
MAX_SAMPLES = None  # None 表示全量，可设为 20 快速验证

samples = test_data[:MAX_SAMPLES] if MAX_SAMPLES else test_data
n = len(samples)

results = []
intent_correct = 0
param_correct = 0

for i in tqdm(range(0, n, BATCH_SIZE), desc='推理中'):
    batch = samples[i:i + BATCH_SIZE]
    infer_requests = []
    
    for s in batch:
        tools = json.loads(s['tools']) if isinstance(s['tools'], str) else s['tools']
        user_msg = next(m for m in s['messages'] if m['role'] == 'user')
        infer_requests.append({
            'tools': tools,
            'messages': [user_msg]
        })
    
    try:
        resp_list = engine.infer(infer_requests)
    except Exception as e:
        print(f'Batch {i} 推理失败: {e}')
        for _ in batch:
            results.append({'intent_ok': False, 'param_ok': False, 'error': str(e)})
        continue
    
    for j, (s, resp) in enumerate(zip(batch, resp_list)):
        content = resp.choices[0].message.content if resp.choices else ''
        pred_name, pred_args = parse_tool_call_from_response(content)
        gt_name, gt_args = get_ground_truth_tool_call(s['messages'])
        
        intent_ok = (pred_name == gt_name) if gt_name else False
        param_ok = args_match(pred_args or {}, gt_args or {}) if intent_ok else False
        
        if intent_ok:
            intent_correct += 1
        if param_ok:
            param_correct += 1
        
        results.append({
            'user': s['messages'][0]['content'][:60],
            'gt_name': gt_name,
            'pred_name': pred_name,
            'intent_ok': intent_ok,
            'param_ok': param_ok,
            'pred_raw': content[:200] if content else ''
        })

print(f'\n完成 {len(results)} 条推理')

## 6. 准确率统计

In [ ]:
total = len(results)
intent_acc = intent_correct / total if total else 0
param_acc = param_correct / total if total else 0

print('=' * 50)
print('graduate_agent_data 测试集 - 原模型基线')
print('=' * 50)
print(f'总样本数: {total}')
print(f'意图识别准确率: {intent_correct}/{total} = {intent_acc:.2%}')
print(f'参数解析准确率: {param_correct}/{total} = {param_acc:.2%}')
print('=' * 50)

## 7. 错误样本查看

In [ ]:
errors = [r for r in results if not r.get('param_ok', True)]
print(f'错误/部分错误样本数: {len(errors)}')
for i, r in enumerate(errors[:5]):
    print(f'\n--- 样本 {i+1} ---')
    print(f'User: {r.get("user", "N/A")}...')
    print(f'GT tool: {r.get("gt_name")}')
    print(f'Pred tool: {r.get("pred_name")}')
    print(f'Intent OK: {r.get("intent_ok")}, Param OK: {r.get("param_ok")}')
    if r.get('error'):
        print(f'Error: {r["error"]}')
    else:
        print(f'Pred raw: {(r.get("pred_raw") or "")[:150]}...')

In [ ]:
# 可选：保存评估结果到文件
# import json
# with open('evaluate_baseline_results.json', 'w', encoding='utf-8') as f:
#     json.dump({
#         'summary': {'intent_acc': intent_acc, 'param_acc': param_acc, 'total': total},
#         'results': results
#     }, f, ensure_ascii=False, indent=2)
# print('已保存到 evaluate_baseline_results.json')